# 🤗 x 🦾: Training SmolVLA with LeRobot Notebook

**LeRobot SmolVLA training notebook**에 오신 것을 환영합니다! 이 노트북은 [🤗 LeRobot](https://github.com/huggingface/lerobot) 라이브러리를 사용하여 모방 학습(imitation learning) 정책을 학습시킬 수 있는 즉시 실행 가능한 환경을 제공합니다.

이 예제에서는 [Hugging Face Hub](https://huggingface.co/)에 호스팅된 데이터셋을 사용하여 `SmolVLA` policy를 train시키고, 선택적으로 [Weights & Biases (wandb)](https://wandb.ai/)를 통해 학습 지표를 추적합니다.

## ⚙️ Requirements
- 학습 데이터가 포함된 Hugging Face 데이터셋 저장소 ID (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- 선택사항: 학습 시각화를 원하는 경우 [wandb](https://wandb.ai/) 계정
- 권장사항: 빠른 학습을 위한 GPU 런타임 (예: NVIDIA A100)

## ⏱️ Expected Training Time
`SmolVLA` policy로 20,000 스텝을 학습하는 데 **NVIDIA A100 GPU 기준 약 5시간이 소요**됩니다. 성능이 낮은 GPU나 CPU에서는 훨씬 더 오래 걸릴 수 있습니다!

## Example Output
모델 체크포인트, 로그, 학습 그래프가 지정된 `--output_dir`에 저장됩니다. `wandb`를 활성화한 경우, wandb 프로젝트 대시보드에서도 진행 상황을 확인할 수 있습니다.


## Install conda

이 셀은 `condacolab`을 사용하여 Google Colab 내부에 완전한 Conda 환경을 구성합니다.


In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## Install LeRobot

이 셀은 Hugging Face에서 `lerobot` repository를 클론하고, FFmpeg(버전 7.1.1)를 설치한 후, 패키지를 편집 가능 모드로 설치합니다.


In [ ]:
!git clone https://github.com/huggingface/lerobot.git
!conda install ffmpeg=7.1.1 -c conda-forge
!cd lerobot && pip install -e .

## Weights & Biases login

이 셀은 실험 추적 및 로깅을 활성화하기 위해 Weights & Biases(wandb)에 로그인합니다.

In [ ]:
!wandb login

## Install SmolVLA dependencies

In [ ]:
!cd lerobot && pip install -e ".[smolvla]"

## Start training SmolVLA with LeRobot

이 셀은 `lerobot` 라이브러리의 `train.py` 스크립트를 실행하여 robot control policy을 학습시킵니다.

다음 인수들을 본인의 설정에 맞게 조정하세요:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`: 데이터셋이 저장된 Hugging Face Hub 저장소 ID로 변경하세요. 예: pepijn223/il_gym0

2. `--batch_size=64`: 모델이 한 번의 그래디언트 업데이트를 수행하기 전에 병렬로 처리하는 학습 샘플 수를 의미합니다. GPU 메모리가 부족한 경우 이 숫자를 줄이세요.

3. `--output_dir=outputs/train/...`: 학습 로그와 모델 체크포인트가 저장될 디렉토리입니다.

4. `--job_name=...`: 이 학습 작업의 이름으로, 로깅 및 Weights & Biases에서 사용됩니다.

5. `--policy.device=cuda`: NVIDIA GPU에서 학습하는 경우 cuda를 사용하세요. Apple Silicon의 경우 mps, GPU가 없는 경우 cpu를 사용하세요.

6. `--wandb.enable=true`: 학습 진행 상황 시각화를 위해 Weights & Biases를 활성화합니다. 실행 전에 wandb login을 통해 로그인해야 합니다.



In [ ]:
!cd lerobot && python lerobot/scripts/train.py \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=${HF_USER}/mydataset \
  --batch_size=64 \
  --steps=20000 \
  --output_dir=outputs/train/my_smolvla \
  --job_name=my_smolvla_training \
  --policy.device=cuda \
  --wandb.enable=true

## Login into Hugging Face Hub
학습이 완료된 후 Hugging Face Hub에 로그인하고 마지막 체크포인트를 업로드합니다.

In [ ]:
!huggingface-cli login

In [ ]:
!huggingface-cli upload ${HF_USER}/my_smolvla \
  /content/lerobot/outputs/train/my_smolvla/checkpoints/last/pretrained_model